O que ? QSPR? 
 
- (Q)SPR (Rela??o Quantitativa Estrutura-Propriedade) 

- Esse modelo SPR desenvolvido corresponde a compreens?o se caso a mol?cula ? mutag?nica ou n?o. 

- Mutagenicidade ? a propriedade de subst?ncias para induzir muta??o gen?tica. ? uma das propriedades ambientais, de sa?de e seguran?a (EHS) mais importantes a serem verificadas ao lidar com novos produtos qu?micos (por exemplo, drogas ou solventes). Neste caso, vamos usar os dados de mutagenicidade em Salmonella typhimurium (teste de Ames). Este conjunto de dados foi coletado pelo Istituto di Ricerche Farmacologiche Mario Negri, mesclando dados experimentais de um conjunto de dados de refer?ncia compilado por [HMS 09] a partir de uma cole??o de dados disponibilizada pelo Minist?rio da Sa?de do Jap?o em seu projeto Ames (Q)SAR.

# 1. Importando as bibliotecas

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

import seaborn as sns
import pandas as pd
import numpy as np

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# 2. Importando dados 

In [ ]:
data = pd.read_csv("https://raw.githubusercontent.com/edgarsmdn/MLCE_book/main/references/mutagenicity_kNN.csv")

In [ ]:
data.head()

In [ ]:
# Tipo dos dados

data.dtypes

In [ ]:
# Nao existe dados nulos

data.isnull().sum() 

In [ ]:
# Acessando os primeiros 4 valores do dataframe por index

primeiras_linhas = data.iloc[:4]
primeiras_linhas

In [ ]:
# Acessando e guardando valores da coluna em espec?fico desejada

y_experimento_result = data['Experimental value']
y_experimento_result

In [ ]:
# Acessando e guardando valores da coluna Valores experimentais com resultados iguais a 1 (positivo)

dados_mutagenicos = data[data['Experimental value'] == 1]
dados_mutagenicos

In [ ]:
data['Predicted value']

In [ ]:
# Verificando os valores dentro da coluna

data['Predicted value'].unique()

In [ ]:
data['Predicted value'].drop_duplicates()

In [ ]:
# Aqui vamos ter acesso os valores que n?o s?o nem "0" e nem "1"
# T?m-se que 6 valores do dataset n?o possuem "Predicted value"

data[(data['Predicted value'] != "0") & (data['Predicted value'] != "1")] 

In [ ]:
# Procurando os dados que n?o possuem valores iguais a "0" e "1" para a coluna "Predicted value"
# E tirando os valores que n?o s?o "0" e "1" com a funcao drop

dados_non_predicted = data[(data['Predicted value'] != "0") & (data['Predicted value'] != "1")].index
data = data.drop(dados_non_predicted)

In [ ]:
data.shape

In [ ]:
data.columns

In [ ]:
# Tirando as colunas desnecess?rias com a funcao drop

colunas_desnecessarias = ['Unnamed: 0', 'Id', 'CAS', 'SMILES', 'Status', 'Experimental value',
       'Predicted value']

data_backup = data
data = data.drop(colunas_desnecessarias, axis=1)

In [ ]:
data.head()

In [ ]:
# Exercicio do curso: How many molecules in our dataset have a qed less than 0.5?

qed_5 = data[data['qed'] < 0.5]
qed_5['qed'].value_counts().sum()

In [ ]:
# Exercicio do curso: What is the molecule with the largest molecular weight MolWt?

data['MolWt'].max()

In [ ]:
# Exercicio do curso: What is the average number of valance electrons NumValenceElectrons of the molecules in our dataset?

data['NumValenceElectrons'].mean()

In [ ]:
X = data

# 3. Feature Engineering

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
scaler = StandardScaler()
scaler.fit(X)

In [ ]:
X_hat = scaler.transform(X)
X_hat # X_hat ? o dataframe j? com o StandardScaler

In [ ]:
X_hat_backup = X_hat

In [ ]:
# Exercicio curso: What are exactly the mean an standard deviation vectors that we used to scaled the data?



In [ ]:
data_backup.head()

In [ ]:
y = data_backup['Experimental value'].to_numpy() # A funcao to_numpy() retorna um array com os valores
mutacao_genetica_perc = (y.sum()/len(y))*100

In [ ]:
print(f"Porcentagem de mol?culas mutag?nicas: {mutacao_genetica_perc}")
print(f"Porcentagem de mol?culas n?o mutag?nicas: {100-mutacao_genetica_perc}")

# 4. Aplicando a Machine Learning

In [ ]:
# importando a biblioteca train_test_split from sklearn.model_selection para aplica??o da machine learning
# dividindo os valores para a machine learning

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_hat, y, test_size=0.2, random_state=0)

In [ ]:
print(f"Y dados de Treino: {y_train.shape}")
print(f"Y dados de Teste:  {y_test.shape}")

In [ ]:
print(f"X dados de Treino: {X_train.shape}")
print(f"X dados de Teste:  {X_test.shape}")

In [ ]:
# importando a biblioteca kNN para aplica??o do modelo

from sklearn.neighbors import KNeighborsClassifier

In [ ]:
knn = KNeighborsClassifier(n_neighbors=3)

In [ ]:
knn.fit(X_train, y_train)

In [ ]:
y_pred = knn.predict(X_test)

# 5. Avalia??o da Machine Learning 

## 5.1. kNN-Neighbors

In [ ]:
from sklearn.metrics import (confusion_matrix, accuracy_score, recall_score, f1_score, roc_auc_score, ConfusionMatrixDisplay, precision_score)

In [ ]:
# Confusion_matrix

cm = confusion_matrix(y_test, y_pred)

sns.heatmap(cm, cmap='OrRd', annot=True, fmt='2.0f')
plt.title('kNN-Neigbors')
plt.ylabel('P R E V I S T O')
plt.xlabel('R E A L')
plt.show()

In [ ]:
print("kNN-Neighbors - Metrics")
print(f"Acur?cia: {accuracy_score(y_test, y_pred).round(2)}")
print(f"Precis?o: {precision_score(y_test, y_pred).round(2)}")
print(f"Recall: {recall_score(y_test, y_pred).round(2)}")
print(f"F1: {f1_score(y_test, y_pred).round(2)}")

## 5.2. VEGA

In [ ]:
data_backup.head()

In [ ]:
y_clean = data_backup['Experimental value'].astype(int)
y_vega_clean = data_backup['Predicted value'].astype(int)

In [ ]:
cm_vega = confusion_matrix(y_clean, y_vega_clean)

In [ ]:
sns.heatmap(cm_vega, cmap='OrRd', annot=True, fmt='2.0f')
plt.title('kNN-Neigbors')
plt.ylabel('P R E V I S T O')
plt.xlabel('R E A L')
plt.show()

In [ ]:
print("VEGA - Metrics")
print(f"Acur?cia: {accuracy_score(y_clean, y_vega_clean).round(2)}")
print(f"Precis?o: {precision_score(y_clean, y_vega_clean).round(2)}")
print(f"Recall: {recall_score(y_clean, y_vega_clean).round(2)}")
print(f"F1: {f1_score(y_clean, y_vega_clean).round(2)}")

As m?tricas finais para os valores produzidos a partir da Machine Learning utilizando vega obtiveram melhores resultados que o kNN-Neighbors

## 5.3. Hyperparameter

In [ ]:
# criando o sistema para hyperpar?metro
# escolhendo um array de valores para k, sendo num_ks um array de 1 a 100, variando de 2 em 2
# seprando os X_train e y_train em X_train_hyp, y_train_hyp, e separando para X_valid e y_valid 
# dentro do for est? testando com os valores de k estipulados dentro do array num_ks que vai dar um novo valor para k


from tqdm.notebook import tqdm

num_ks = np.arange(1, 100, 2).astype(int)

X_train_hyp, X_valid, y_train_hyp, y_valid = train_test_split(X_train, y_train, test_size=0.1, random_state=0)

train_accuracy = []
valid_accuracy = []

for i in tqdm(range(len(num_ks))):
    knn = KNeighborsClassifier(n_neighbors=num_ks[i])
    knn.fit(X_train_hyp, y_train_hyp)
    
    pred_train = knn.predict(X_train_hyp)
    pred_valid  = knn.predict(X_valid)
    
    train_accuracy.append(1-accuracy_score(y_train_hyp, pred_train))
    valid_accuracy.append(1-accuracy_score(y_valid, pred_valid))

In [ ]:
plt.figure(figsize=(10,6), dpi=200)

plt.plot(num_ks, train_accuracy, 'bs--', label="Treinamento")
plt.plot(num_ks, valid_accuracy, 'rx--', label="Valida??o")

plt.xlabel('Valor para k')
plt.ylabel('Taxa de classifica??o incorreta')

plt.legend()
plt.show()

## 5.4. Cross-Validation

In [ ]:
from sklearn.model_selection import cross_validate

In [ ]:
num_ks = np.arange(1, 50, 1).astype(int)

train_misclassification = []
valid_misclassification = []

for i in tqdm(range(len(num_ks))):
    knn = KNeighborsClassifier(n_neighbors=num_ks[i])
    cv_dict = cross_validate(knn, X_train, y_train, cv=10, scoring='accuracy', return_train_score=True)
    
    k_fold_train_scored = cv_dict['train_score']
    k_fold_valid_scored = cv_dict['test_score']
    
    train_misclassification.append(1-k_fold_train_scored.mean())
    valid_misclassification.append(1-k_fold_valid_scored.mean())

In [ ]:
plt.figure(figsize=(10,6), dpi=200)

plt.plot(num_ks, train_misclassification, 'bs--', label='Treinamento')
plt.plot(num_ks, valid_misclassification, 'rx--', label='Valida??o')

plt.xlabel('Valor para k')
plt.ylabel('Taxa de classifica??o incorreta')

plt.legend()
plt.show()

In [ ]:
print('N? k com erro de classifica??o m?nimo de valida??o: ', num_ks[np.argmin(valid_misclassification)])

In [ ]:
plt.figure(figsize=(10,6))

plt.plot(num_ks, valid_misclassification, 'rx--', label='Valida??o')

plt.yscale('log')
plt.xlabel('N? de k')
plt.ylabel('Taxa de classifica??o incorreta')

plt.legend()
plt.show()

Exercicio: You can now go back to your implementation of leave-one-out and train your kNN model using the best k that you found. How does it perform compared to the VEGA kNN? Do you got a better model? Why?

In [ ]:
knn = KNeighborsClassifier(n_neighbors=17)

knn.fit(X_train, y_train)

y_pred_new_k = knn.predict(X_test)

In [ ]:
print("kNN-Neighbors - Metrics")
print(f"Acur?cia: {accuracy_score(y_test, y_pred_new_k).round(2)}")
print(f"Precis?o: {precision_score(y_test, y_pred_new_k).round(2)}")
print(f"Recall: {recall_score(y_test, y_pred_new_k).round(2)}")
print(f"F1: {f1_score(y_test, y_pred_new_k).round(2)}")

N?o, pois comparado aos valores demonstrados a cima, quanto maiores os valores para k mais a taxa de classifica??o incorreta tende a crescer, dessa forma o intervalo entre os valores 10 e 20 possuem as menores taxas de classifica??o incorreta. 